# Promedios del jugador sin fuga de datos

## Qué responde este notebook

Los notebooks originales del proyecto (`weekly/wr/notebooks/_reference/`) calculaban mal el promedio
de carrera de cada jugador: el resultado quedaba desalineado con sus filas y, en el primer partido
de casi todos los jugadores, aparecía un promedio que todavía no podía existir. Es una fuga de datos:
el modelo veía información que no estaba disponible antes del partido (ver
[`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md) y
[`docs/decisions/0001-criterios-de-replica.md`](../../../docs/decisions/0001-criterios-de-replica.md)).

La pregunta es si el código nuevo, `weekly/wr/src/features.py`, evita ese error, comprobado fila por
fila con un jugador real.

## Qué calcula `features.py`

Para cada jugador y cada estadística (por ejemplo, yardas de recepción) calcula promedios con **solo
los partidos anteriores**, nunca el de la semana misma (definiciones en el
[glosario](../../../docs/data/GLOSARIO_VARIABLES.md), sección 7.2):

- **`_season_avg`**: promedio de la temporada en curso hasta la semana anterior. Se reinicia en cada
  temporada.
- **`_last3_avg` y `_last5_avg`**: promedio de las últimas 3 y de las últimas 5 apariciones del
  jugador, sin reiniciar por temporada. Se calculan las dos; cuál aporta más se evalúa en
  [`3.2_seleccion_features.ipynb`](3.2_seleccion_features.ipynb).
- **`_career_avg`**: promedio de toda su carrera hasta la semana anterior.

`target_share`, `air_yards_share` y `wopr` no se calculan aquí: ya vienen en la tabla de
estadísticas (`cargar_stats_semanales()`, ver
[`1.2_construccion_datos.ipynb`](1.2_construccion_datos.ipynb)), mientras que los notebooks
originales los recalculaban a partir de las jugadas.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features

stats = data.cargar_stats_semanales([2024, 2025])
wr = stats[stats["position"] == "WR"].copy()

resultado = features.agregar_promedios_jugador(wr, ["receiving_yards"], id_col="player_id")  # ventanas=(3,5) por default
print("filas:", resultado.shape[0])

filas: 4952


In [2]:
resultado

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,...,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr,receiving_yards_season_avg,receiving_yards_last3_avg,receiving_yards_last5_avg,receiving_yards_career_avg
26,00-0030035,A.Thielen,Adam Thielen,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2024,1,REG,2024_01_CAR_NO,...,0,0,0,0,4.9,7.9,NaN,NaN,NaN,NaN
1069,00-0030035,A.Thielen,Adam Thielen,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2024,2,REG,2024_02_LAC_CAR,...,0,0,0,0,2.0,4.0,49.000000,49.000000,49.000000,49.000000
2118,00-0030035,A.Thielen,Adam Thielen,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2024,3,REG,2024_03_CAR_LV,...,0,0,0,0,10.0,13.0,34.500000,34.500000,34.500000,34.500000
11031,00-0030035,A.Thielen,Adam Thielen,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2024,12,REG,2024_12_KC_CAR,...,0,0,0,0,5.7,8.7,36.333333,36.333333,36.333333,36.333333
11905,00-0030035,A.Thielen,Adam Thielen,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2024,13,REG,2024_13_TB_CAR,...,0,0,0,0,15.9,23.9,41.500000,39.000000,41.500000,41.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32384,00-0040782,I.Bond,Isaiah Bond,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2025,14,REG,2025_14_TEN_CLE,...,0,0,0,0,0.0,0.0,19.900000,25.333333,17.000000,19.900000
33439,00-0040782,I.Bond,Isaiah Bond,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2025,15,REG,2025_15_CLE_CHI,...,0,0,0,0,8.9,10.9,18.090909,25.333333,15.200000,18.090909
34537,00-0040782,I.Bond,Isaiah Bond,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2025,16,REG,2025_16_BUF_CLE,...,0,0,0,0,2.0,2.0,24.000000,35.666667,33.000000,24.000000
35601,00-0040782,I.Bond,Isaiah Bond,WR,WR,https://static.www.nfl.com/image/upload/f_auto...,2025,17,REG,2025_17_PIT_CLE,...,0,0,0,0,1.6,2.6,23.769231,36.666667,37.200000,23.769231


## La prueba: un jugador real, semana por semana

Un receptor con muchas semanas en 2024 y 2025 (DeVonta Smith), revisado fila por fila: el promedio de
cada semana debe usar solo las anteriores.

In [3]:
smith = resultado[resultado["player_display_name"] == "DeVonta Smith"].sort_values(["season", "week"])
cols = ["season", "week", "receiving_yards", "receiving_yards_season_avg",
        "receiving_yards_last3_avg", "receiving_yards_last5_avg", "receiving_yards_career_avg"]
smith[cols].head(9)

,season,week,receiving_yards,receiving_yards_season_avg,receiving_yards_last3_avg,receiving_yards_last5_avg,receiving_yards_career_avg
595,2024,1,84,NaN,NaN,NaN,NaN
1639,2024,2,76,84.000000,84.000000,84.000000,84.000000
2685,2024,3,79,80.000000,80.000000,80.000000,80.000000
5659,2024,6,64,79.666667,79.666667,79.666667,79.666667
6656,2024,7,-2,75.750000,73.000000,75.750000,75.750000
7697,2024,8,85,60.200000,47.000000,60.200000,60.200000
8716,2024,9,87,64.333333,49.000000,60.400000,64.333333
9675,2024,10,14,67.571429,56.666667,62.600000,67.571429
10585,2024,11,29,60.875000,62.000000,49.600000,60.875000


**Lo que muestran las filas:**

- **Semana 1 de 2024**: todos los promedios están vacíos, porque no hay ningún partido anterior. Si
  aquí hubiera un número, habría fuga.
- **Semana 2**: `season_avg` = 84.0, exactamente las yardas de la semana 1.
- **Semana 7** (−2 yardas esa semana): `season_avg` = 75.75, el promedio de sus partidos anteriores
  de la temporada (84, 76, 79 y 64, de las semanas 1, 2, 3 y 6; no jugó la 4 ni la 5). No incluye el
  −2 de la semana misma. `last3_avg` = 73.0, el promedio de 76, 79 y 64.
- **Semana 8**: `last3_avg` = 47.0, el promedio de 79, 64 y −2. Ahora sí entra el −2, porque la
  semana 7 ya es un partido anterior.

Cada valor se puede recalcular a mano. En la semana 8 también se ve por qué conviene tener las dos
ventanas: `last3_avg` = 47.0 y `last5_avg` = 60.2, porque el −2 pesa más en la ventana corta.

## La segunda prueba: ¿`season_avg` se reinicia y `career_avg` no?

Al pasar de 2024 a 2025, `season_avg` debe empezar de cero y `career_avg` debe seguir acumulando.

In [4]:
smith[cols].tail(5)

,season,week,receiving_yards,receiving_yards_season_avg,receiving_yards_last3_avg,receiving_yards_last5_avg,receiving_yards_career_avg
31844,2025,14,37,66.833333,48.333333,59.6,65.400000
32838,2025,15,50,64.538462,58.000000,50.2,64.307692
33905,2025,16,42,63.500000,45.000000,46.4,63.777778
34993,2025,17,25,62.066667,43.000000,53.2,63.000000
36038,2025,18,52,59.750000,39.000000,40.4,61.689655


Aquí `career_avg` solo cubre 2024-2025, las temporadas cargadas, así que si `season_avg` no se
reiniciara sería exactamente igual a `career_avg`. En la última fila (semana 18 de 2025) son
distintos: 59.75 contra 61.69. En la tabla de modelado, `career_avg` cubre desde 2016.

## Por qué el código no puede repetir el error

En los notebooks originales, el promedio se calculaba con `groupby().apply()` y se pegaba de vuelta
con `.reset_index(drop=True)`: el resultado queda en el orden de los grupos y el índice nuevo ya no
corresponde a las filas originales, así que cada fila recibe el promedio de otra (ver
[`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)).

`agregar_promedios_jugador()` usa `groupby(...).transform(...)`, que regresa el resultado alineado
al índice original de cada fila, sin ningún paso manual de reacomodo donde ese error pueda ocurrir.

## Cierre

- Los promedios de `features.py` usan solo partidos anteriores: la semana 1 queda vacía, cada
  promedio excluye la semana misma y la ventana avanza como se espera.
- `season_avg` se reinicia cada temporada y `career_avg` sigue acumulando.
- `transform` mantiene cada promedio alineado con su fila, que es justo el paso donde fallaban los
  notebooks originales.

Cuál de las ventanas aporta más, y si conviene agregar variables de equipo, se evalúa en
[`3.1_ingenieria_features.ipynb`](3.1_ingenieria_features.ipynb) y
[`3.2_seleccion_features.ipynb`](3.2_seleccion_features.ipynb).

Código: [`../src/features.py`](../src/features.py) · Anterior:
[`1.3_unificacion_alineaciones.ipynb`](1.3_unificacion_alineaciones.ipynb) · Siguiente:
[`2.1_eda_targets.ipynb`](2.1_eda_targets.ipynb)